In [1]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens
import random
from torch.nn.utils.rnn import pad_sequence

import matplotlib.pyplot as plt

from protgen.models import MLP, LSTM, ClassifierLSTM

In [2]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [3]:
print(f"vocab ({train_data.vocab_size} tokens): {train_data.vocab}")
print(f"families: {train_data.fam_to_idx}")
print(f"padding is ignored in the loss with ignore_index={IGNORE_INDEX}")

sequence = train_data.df["sequence"][0]
tokens = train_data.encode(sequence)
print()
print(f"encode / decode round-trips: {train_data.decode(tokens) == sequence}")
print(f"and from the shifted target too:  {train_data.decode(x_shifted) == sequence}")

vocab (23 tokens): {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, 'A': 3, 'C': 4, 'D': 5, 'E': 6, 'F': 7, 'G': 8, 'H': 9, 'I': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'V': 20, 'W': 21, 'Y': 22}
families: {'AcyP': 0, 'HPr': 1, 'HU': 2}
padding is ignored in the loss with ignore_index=-1

encode / decode round-trips: True
and from the shifted target too:  True


In [4]:
x, x_shifted, y, length = next(iter(train_loader))
print(f"batch: x {tuple(x.shape)} {x.dtype}, x_shifted {tuple(x_shifted.shape)}, y {tuple(y.shape)}")

# Generative: one distribution over the vocabulary per position.
logits_gen = torch.randn(x.shape[0], train_data.vocab_size, train_data.max_length)
print(f"generative loss:     {F.cross_entropy(logits_gen, x_shifted, ignore_index=IGNORE_INDEX):.4f}")

# Classification: one distribution over the three families per sequence.
logits_fam = torch.randn(x.shape[0], len(FAMILIES))
print(f"classification loss: {F.cross_entropy(logits_fam, y):.4f}")

batch: x (32, 129) torch.int64, x_shifted (32, 129), y (32,)
generative loss:     3.5973
classification loss: 1.3327


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
with open(f"../src/protgen/models/LSTM_config.json", "r", encoding="utf-8") as f:
    generator_config = json.load(f)

generator = LSTM.LSTMModel(
    vocab_size = generator_config["vocab_size"],
    embed_dim = generator_config["embed_dim"], 
    hidden_dim = generator_config["hidden_dim"],
    num_layers = generator_config["num_layers"],
    dropout = generator_config["dropout"]
)

generator.load_state_dict(
    torch.load("../src/protgen/models/best_LSTM_model.pt", map_location = device, weights_only = False)
)
generator.to(device)

LSTMModel(
  (embedding): Embedding(23, 32)
  (rnn): LSTM(32, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=128, out_features=23, bias=True)
)

In [9]:
with open(f"../src/protgen/models/ClassifierLSTM_config.json", "r", encoding="utf-8") as f:
    classifier_config = json.load(f)

classifier = ClassifierLSTM.NoisyPredictor(
    vocab_size = classifier_config["vocab_size"],
    embed_dim = classifier_config["embed_dim"],
    hidden_dim = classifier_config["hidden_dim"],
    num_layers = classifier_config["num_layers"],
    num_classes = classifier_config["num_classes"],
    dropout = classifier_config["dropout"]
)

classifier.load_state_dict(
    torch.load("../src/protgen/models/best_ClassifierLSTM_model.pt", map_location = device, weights_only = False)
)
classifier.to(device)

NoisyPredictor(
  (embedding): Embedding(23, 32)
  (rnn): LSTM(32, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

In [10]:
vocab_size = train_data.vocab_size
num_classes = len(train_data.fam_to_idx)
pad_id = 0
bos_id = 1
eos_id = 2

In [11]:
@torch.no_grad()
def generate_guided(generator, classifier, target_family, lambda_guidance=1.0, max_length=128, device="cuda"):
    generator.eval()
    classifier.eval()

    # Start with BOS
    sequence = torch.tensor([[bos_id]], dtype=torch.long, device=device)
    aa_ids = torch.arange(3, vocab_size, device=device)

    for step in range(max_length):
        # 1. Generator predicts next-token logits
        logits = generator(sequence)
        next_logits = logits[:, -1, :].clone()

        # Never generate PAD or BOS
        next_logits[:, pad_id] = -float("inf")
        next_logits[:, bos_id] = -float("inf")

        # 2. Apply classifier guidance
        if lambda_guidance > 0:
            # Construct a prefix for each candidate amino acid
            candidates = sequence.repeat(len(aa_ids), 1)
            candidates = torch.cat([candidates, aa_ids.unsqueeze(1)], dim=1)
            lengths = torch.full((len(aa_ids),), candidates.size(1), dtype=torch.long, device=device)

            # 3. Predict target family probability
            class_logits = classifier(candidates, lengths)
            log_probs = F.log_softmax(class_logits, dim=-1)
            target_log_probs = log_probs[:, target_family]

            # 4. Reweight generator logits
            next_logits[0, aa_ids] += (lambda_guidance * target_log_probs)
            
        # 5. Sample from guided distribution
        probabilities = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)

        # Stop on EOS
        if next_token.item() == eos_id:
            break

        sequence = torch.cat([sequence, next_token], dim=1)

    return sequence.squeeze(0)

In [12]:
target_family = train_data.fam_to_idx["AcyP"]

generated = generate_guided(
    generator=generator,
    classifier=classifier,
    target_family=target_family,
    lambda_guidance=2.0,
    max_length=128,
    device=device
)

protein = train_data.decode(generated.cpu())

print("Generated protein:")
print(protein)
print("Length:", len(protein))

Generated protein:
MSSVRVAAVRVEGVVQGFGFRYWKRVERDDVGITGWLNNPDGTVVSVAAPGAEAVACLALRGWPAGPATVVAELRVDTGTVHHGGGFQVGLE
Length: 92
